In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [2]:
from forecasting_utils import (
    FOCUS_AGES,
    build_daily_arrivals,
    build_model_frame,
    clean_opossum_data,
    feature_columns,
    save_csv,
)

In [3]:
from dataclasses import dataclass
@dataclass(frozen=True)
class SplitData:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame

In [4]:
def split_for_2026_forecast(df: pd.DataFrame, validation_start: str = "2025-01-01", test_start: str = "2026-01-01") -> SplitData:
    validation_start_ts = pd.Timestamp(validation_start)
    test_start_ts = pd.Timestamp(test_start)

    train = df.loc[df["admission_date"] < validation_start_ts].copy()
    validation = df.loc[
        (df["admission_date"] >= validation_start_ts)
        & (df["admission_date"] < test_start_ts)
        ].copy()
    test = df.loc[df["admission_date"] >= test_start_ts].copy()

    return SplitData(train=train, validation=validation, test=test)

# Preprocessing for Forecasting  

This is where I clean things up and turn the raw intake records into something the forecasting notebooks can actually use.

The goal here was to keep the prep practical: standardize the fields, focus on the age groups I care about, and build a daily arrivals series without overcomplicating it.


In [5]:
df = clean_opossum_data()
focus_df = df.loc[df["is_focus_age"]].copy()
focus_df.info()

<class 'pandas.DataFrame'>
Index: 8607 entries, 21 to 10511
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              8607 non-null   str           
 1   name            5256 non-null   str           
 2   admission_date  8607 non-null   datetime64[us]
 3   status          8607 non-null   string        
 4   final_date      8606 non-null   datetime64[us]
 5   age_raw         8607 non-null   string        
 6   city            8607 non-null   string        
 7   age_group       8607 non-null   string        
 8   year            8607 non-null   int64         
 9   month           8607 non-null   int64         
 10  day             8607 non-null   int64         
 11  day_of_week     8607 non-null   str           
 12  week_of_year    8607 non-null   int64         
 13  days_in_care    8606 non-null   float64       
 14  is_focus_age    8607 non-null   bool          
dtypes: bool(1), dateti

In [6]:
focus_df[["admission_date", "age_raw", "age_group", "city"]].head()

,admission_date,age_raw,age_group,city
21,2000-04-01,I,immature,Lombard
22,2000-04-01,I,immature,Lombard
23,2000-04-01,I,immature,Lombard
24,2000-04-01,I,immature,Lombard
25,2000-04-01,I,immature,Lombard


In [7]:
daily_df = build_daily_arrivals(df, focus_ages=FOCUS_AGES)
daily_df.head()

,admission_date,arrivals,year,quarter,month,day,day_of_week,day_of_year,week_of_year,is_weekend,is_holiday,month_sin,month_cos,dow_sin,dow_cos
0,2000-04-01,10,2000,2,4,1,5,92,13,1,0,0.866025,-0.5,-0.974928,-0.222521
1,2000-04-02,0,2000,2,4,2,6,93,13,1,0,0.866025,-0.5,-0.781831,0.623490
2,2000-04-03,0,2000,2,4,3,0,94,14,0,0,0.866025,-0.5,0.000000,1.000000
3,2000-04-04,0,2000,2,4,4,1,95,14,0,0,0.866025,-0.5,0.781831,0.623490
4,2000-04-05,0,2000,2,4,5,2,96,14,0,0,0.866025,-0.5,0.974928,-0.222521


In [8]:
daily_df.describe().T

,count,mean,min,25%,50%,75%,max,std
admission_date,9507,2013-04-06 00:00:00,2000-04-01 00:00:00,2006-10-03 12:00:00,2013-04-06 00:00:00,2019-10-08 12:00:00,2026-04-11 00:00:00,NaN
arrivals,9507.0,0.905333,0.0,0.0,0.0,0.0,32.0,2.548556
year,9507.0,2012.762701,2000.0,2006.0,2013.0,2019.0,2026.0,7.521584
quarter,9507.0,2.508047,1.0,2.0,3.0,4.0,4.0,1.116605
month,9507.0,6.520248,1.0,4.0,7.0,10.0,12.0,3.447891
day,9507.0,15.717682,1.0,8.0,16.0,23.0,31.0,8.802042
day_of_week,9507.0,3.00021,0.0,1.0,3.0,5.0,6.0,2.000105
day_of_year,9507.0,183.043757,1.0,92.0,183.0,274.0,366.0,105.431741
week_of_year,9507.0,26.558746,1.0,14.0,27.0,40.0,53.0,15.0518
is_weekend,9507.0,0.285789,0.0,0.0,0.0,1.0,1.0,0.451813


### Create lagged and rolling features

The models will use:

- calendar features
- holiday and weekend indicators
- short-term lags
- rolling summary statistics


In [9]:
model_df = build_model_frame(df, focus_ages=FOCUS_AGES)
model_df.head()

,admission_date,arrivals,year,quarter,month,day,day_of_week,day_of_year,week_of_year,is_weekend,...,lag_28,rolling_mean_7,rolling_std_7,rolling_max_7,rolling_mean_14,rolling_std_14,rolling_max_14,rolling_mean_28,rolling_std_28,rolling_max_28
0,2000-04-29,0,2000,2,4,29,5,120,17,1,...,10.0,0.571429,1.133893,3.0,1.071429,2.055547,6.0,0.892857,2.346618,10.0
1,2000-04-30,0,2000,2,4,30,6,121,17,1,...,0.0,0.142857,0.377964,1.0,1.071429,2.055547,6.0,0.535714,1.527092,6.0
2,2000-05-01,0,2000,2,5,1,0,122,18,0,...,0.0,0.000000,0.000000,0.0,0.642857,1.499084,5.0,0.535714,1.527092,6.0
3,2000-05-02,5,2000,2,5,2,1,123,18,0,...,0.0,0.000000,0.000000,0.0,0.642857,1.499084,5.0,0.535714,1.527092,6.0
4,2000-05-03,1,2000,2,5,3,2,124,18,0,...,0.0,0.714286,1.889822,5.0,0.642857,1.499084,5.0,0.714286,1.739671,6.0


In [10]:
model_df.isna().sum().sort_values(ascending=False).head(10)

admission_date    0
arrivals          0
year              0
quarter           0
month             0
day               0
day_of_week       0
day_of_year       0
week_of_year      0
is_weekend        0
dtype: int64

In [11]:
feature_cols = feature_columns(model_df)
feature_cols

['year',
 'quarter',
 'month',
 'day',
 'day_of_week',
 'day_of_year',
 'week_of_year',
 'is_weekend',
 'is_holiday',
 'month_sin',
 'month_cos',
 'dow_sin',
 'dow_cos',
 'lag_1',
 'lag_7',
 'lag_14',
 'lag_28',
 'rolling_mean_7',
 'rolling_std_7',
 'rolling_max_7',
 'rolling_mean_14',
 'rolling_std_14',
 'rolling_max_14',
 'rolling_mean_28',
 'rolling_std_28',
 'rolling_max_28']

In [12]:
splits = split_for_2026_forecast(model_df)
split_summary = pd.DataFrame(
    {
        "rows": [len(splits.train), len(splits.validation), len(splits.test)],
        "start": [
            splits.train["admission_date"].min(),
            splits.validation["admission_date"].min(),
            splits.test["admission_date"].min(),
        ],
        "end": [
            splits.train["admission_date"].max(),
            splits.validation["admission_date"].max(),
            splits.test["admission_date"].max(),
        ],
    },
    index=["train", "validation_2025", "test_2026_ytd"],
)
split_summary

,rows,start,end
train,9013,2000-04-29,2024-12-31
validation_2025,365,2025-01-01,2025-12-31
test_2026_ytd,101,2026-01-01,2026-04-11


In [13]:
save_csv(focus_df, "focus_age_records.csv")
save_csv(daily_df, "focus_age_daily_arrivals.csv")
save_csv(model_df, "focus_age_model_frame.csv")

WindowsPath('C:/Users/Owner/PycharmProjects/opos_prediction/data/processed/focus_age_model_frame.csv')